# NeurIPS - Open Polymer Prediction 2025
Predicting polymer properties with machine learning to accelerate sustainable materials research.

https://www.kaggle.com/competitions/neurips-open-polymer-prediction-2025

## Prepare data

In [1]:
import sys, os

module_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if module_path not in sys.path:
    sys.path.insert(0, module_path)

from fedotllm.main import FedotAI
from fedotllm.handlers import JupyterOutput
from fedotllm.llm import AIInference
from fedotllm.utils.kaggle import download_from_kaggle, submit_to_kaggle

competition_name = "neurips-open-polymer-prediction-2025"
dataset_path = os.path.join(os.getcwd(), "competition")

# Define paths
source_dir = dataset_path
base_dir = os.getcwd()
targets = ["Tg", "FFV", "Tc", "Density", "Rg"]

/home/stas/Documents/GitHub/FEDOT.LLM/.venv/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


2025-08-03 16:42:46,559 - HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"


In [2]:
download_from_kaggle(competition_name=competition_name, save_path=dataset_path)

Dataset downloaded and extracted to /home/stas/Documents/GitHub/FEDOT.LLM/experiments/polymers_predict/competition


In [3]:
import os
import pandas as pd

# Load files once
train_df = pd.read_csv(os.path.join(source_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(source_dir, "test.csv"))
sample_df = pd.read_csv(os.path.join(source_dir, "sample_submission.csv"))

# Loop through each target
for target in targets:
    # Create output directory
    target_dir = os.path.join(base_dir, f"polymer_{target}", "competition",)
    os.makedirs(target_dir, exist_ok=True)

    # Copy test file as-is
    test_df.to_csv(os.path.join(target_dir, "test.csv"), index=False)

    # Prepare sample_submission with only current target
    sample_target_df = sample_df[["id", target]]
    sample_target_df.to_csv(os.path.join(target_dir, "sample_submission.csv"), index=False)

    # Prepare train file: only rows where target is not null, and only Id + target columns
    train_target_df = train_df[["id","SMILES", target]].dropna(subset=[target])
    train_target_df.to_csv(os.path.join(target_dir, "train.csv"), index=False)

In [4]:
import shutil
output_path = os.path.join(os.getcwd(), 'output')
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path, exist_ok=True)

## Run FEDOT.LLM

In [5]:
prop_descriptions_dict = {
    "Density": "density", 
    "Tc": "response to heat thermal conductivity(Tc)", 
    "Tg": "glass transition temperature(Tg)", 
    "Rg": "fundamental molecular size and packing efficiency radius of gyration(Rg)", 
    "FFV": "fractional free volume(FFV)",
}

def task_description(target_description):
    return f"""
Your Goal:
Your mission is to predict a polymer's real-world performance directly from its chemical structure. 
You'll be provided with a polymer's structure as a simple text string (SMILES), and your challenge 
is to build a model that can accurately forecast five key metrics that determine how it will behave. 
The {target} is the value to predict.
"""

target = "Density"
target_description = prop_descriptions_dict[target]
description = task_description(target_description)
output_path = os.path.join(base_dir, f"polymer_{target}", "competition")

In [8]:
chem_preset = os.path.join(module_path,"fedotllm/configs/chem.yaml")

fedot_ai = FedotAI(
        task_path=dataset_path,
        #presets=[chem_preset],
        workspace=output_path,
        handlers=JupyterOutput().subscribe
    )

async for _ in fedot_ai.ask(message=description):
    continue

2025-08-03 17:18:23,336 - Config path resolved: default
2025-08-03 17:18:23,337 - Loading default config from: /home/stas/Documents/GitHub/FEDOT.LLM/fedotllm/configs/default.yaml


2025-08-03 17:18:23,345 - FEDOTLLM - INFO - FedotAI ask called with message (first 100 chars): '
Your Goal:
Your mission is to predict a polymer's real-world performance directly from its chemical...'


2025-08-03 17:18:23,345 - FedotAI ask called with message (first 100 chars): '
Your Goal:
Your mission is to predict a polymer's real-world performance directly from its chemical...'


2025-08-03 17:18:23,345 - FEDOTLLM - INFO - TranslatorAgent initialized with provided AIInference instance.


2025-08-03 17:18:23,345 - TranslatorAgent initialized with provided AIInference instance.


...

2025-08-03 17:18:23,350 - TranslatorAgent: Source language for input set to: en


2025-08-03 17:18:23,351 - FEDOTLLM - INFO - Input is already English. No translation needed.


2025-08-03 17:18:23,351 - Input is already English. No translation needed.


2025-08-03 17:18:23,352 - FEDOTLLM - INFO - Input message translated to (first 100 chars): '
Your Goal:
Your mission is to predict a polymer's real-world performance directly from its chemical...'


2025-08-03 17:18:23,352 - Input message translated to (first 100 chars): '
Your Goal:
Your mission is to predict a polymer's real-world performance directly from its chemical...'
